In [ ]:
import torch
import torch.nn.functional as F
import json
import sqlite3
import re
import time
from typing import List, Dict, Any, Tuple, Callable

# 채점 대상 함수들을 포함한 학생의 코드를 import 한다고 가정하거나,
# 여기서는 테스트를 위해 학생의 함수가 정의되어 있다고 전제합니다.
# 실제 환경에서는 'from assignment_student import *' 형태로 사용합니다.

def run_grader(student_module):
    scores = {}
    feedback = {}

    def grade(prob_id, success, message=""):
        scores[prob_id] = 10 if success else 0
        feedback[prob_id] = "PASS" if success else f"FAIL: {message}"

    print("🚀 자동 채점 시스템을 시작합니다...\n")

    # [문제 1 채점] Tokenizer
    try:
        sents = [[10, 20], [30]]
        max_l = 5
        res_ids, res_masks = student_module.preprocess_sentences(sents, max_l)

        cond1 = res_ids.shape == (2, 5)
        cond2 = res_ids[0].tolist() == [1, 10, 20, 2, 0] # BOS + seq + EOS + PAD
        cond3 = res_masks[1].tolist() == [1, 1, 1, 0, 0] # [BOS, 30, EOS, PAD, PAD]
        grade(1, cond1 and cond2 and cond3, "BOS/EOS 위치 또는 Padding 처리가 틀립니다.")
    except Exception as e:
        grade(1, False, str(e))

    # [문제 2 채점] Multi-Head Split
    try:
        t = torch.randn(2, 4, 16) # B=2, S=4, H=16
        res = student_module.split_heads(t, num_heads=4)
        cond1 = res.shape == (2, 4, 4, 4) # (B, nH, S, D)
        # 차원 변환 시 데이터 순서가 보존되었는지 간접 확인 (첫 번째 헤드의 첫 토큰)
        expected_first = t[0, 0, :4]
        cond2 = torch.allclose(res[0, 0, 0, :], expected_first)
        grade(2, cond1 and cond2, "텐서 Shape 변환 또는 데이터 보존 로직이 틀립니다.")
    except Exception as e:
        grade(2, False, str(e))

    # [문제 3 채점] KV Cache Concatenation
    try:
        pk = torch.zeros(1, 1, 4, 16)
        nk = torch.ones(1, 1, 1, 16)
        res = student_module.append_to_kv_cache(pk, nk)
        cond = res.shape == (1, 1, 5, 16) and res[0,0,4,0] == 1.0
        grade(3, cond, "Sequence 차원 결합 로직이 틀립니다.")
    except Exception as e:
        grade(3, False, str(e))

    # [문제 4 채점] Temperature Softmax
    try:
        logits = torch.tensor([1.0, 2.0, 10.0])
        # 낮은 온도는 확률을 한곳으로 몰아야 함
        p_low = student_module.get_probs_with_temp(logits, 0.1)
        p_high = student_module.get_probs_with_temp(logits, 10.0)
        cond1 = torch.isclose(p_low.sum(), torch.tensor(1.0))
        cond2 = p_low[2] > p_high[2] # 10.0인 인덱스의 확률이 온도가 낮을 때 더 높아야 함
        grade(4, cond1 and cond2, "Temperature 적용 공식 또는 Softmax 차원이 틀립니다.")
    except Exception as e:
        grade(4, False, str(e))

    # [문제 5 채점] JSON Schema
    try:
        schema = student_module.currency_tool_schema
        props = schema.get("properties", {})
        cond1 = "amount" in schema.get("required", [])
        cond2 = props.get("to_cur", {}).get("default") == "USD"
        grade(5, cond1 and cond2, "Schema의 필수 인자 또는 기본값 설정이 누락되었습니다.")
    except Exception as e:
        grade(5, False, str(e))

    # [문제 6 채점] Arg Parser
    try:
        call = "CALL: order(item=GPU, qty=2)"
        res = student_module.parse_model_call(call)
        cond = res["func_name"] == "order" and res["args"]["item"] == "GPU"
        grade(6, cond, "정규표현식 파싱 결과가 딕셔너리 구조와 일치하지 않습니다.")
    except Exception as e:
        grade(6, False, str(e))

    # [문제 7 채점] MHA KV Update
    try:
        pk = torch.zeros(1, 2, 2, 8)
        nk_p = torch.ones(1, 1, 16)
        res = student_module.update_mha_kv_cache(pk, nk_p, 2)
        cond = res.shape == (1, 2, 3, 8) and torch.all(res[:, :, 2, :] == 1.0)
        grade(7, cond, "MHA 차원 분리 후 결합 로직이 틀립니다.")
    except Exception as e:
        grade(7, False, str(e))

    # [문제 8 채점 - 킬러 1] Multi-Step Chaining
    try:
        tools = {
            "find_product_id": lambda n: "p1" if n=="Phone" else None,
            "get_discount_rate": lambda pid: 0.2 if pid=="p1" else 0.6,
            "calculate_price": lambda pid, r: int(100 * (1-r))
        }
        res1 = student_module.multi_step_tool_agent("Phone", tools)
        res2 = student_module.multi_step_tool_agent("Desk", tools)

        cond1 = "Final Price for Phone is 80" in res1
        cond2 = res2 == "ERROR: PRODUCT_NOT_FOUND"
        grade(8, cond1 and cond2, "체이닝 로직 혹은 에러 메시지 형식이 불일치합니다.")
    except Exception as e:
        grade(8, False, str(e))

    # [문제 9 채점] Sliding Window
    try:
        cache = torch.arange(16).view(1, 1, 4, 4).float() # 0~15
        new_v = torch.full((1, 1, 1, 4), 99.0)
        res = student_module.sliding_window_batch_kv(cache, new_v, 4)
        cond1 = res.shape == (1, 1, 4, 4)
        cond2 = res[0, 0, 0, 0] == 4.0 # 0번 인덱스(값 0~3)가 밀려나고 1번 인덱스(값 4)가 0번으로 와야 함
        grade(9, cond1 and cond2, "Sliding Window의 FIFO(First-In-First-Out) 로직이 틀립니다.")
    except Exception as e:
        grade(9, False, str(e))

    # [문제 10 채점 - 킬러 2] API Retry
    try:
        def mock_api_fail(p): return (429, "Limit")
        def mock_api_server_err(p): return (500, "Error")

        res_limit = student_module.call_llm_api_safely("Prompt", 10, mock_api_fail)
        res_server = student_module.call_llm_api_safely("Prompt", 10, mock_api_server_err)
        res_token = student_module.call_llm_api_safely("Too many words here", 2, mock_api_fail)

        cond1 = res_limit.get("reason") == "RATE_LIMIT_EXCEEDED"
        cond2 = res_server.get("reason") == "SERVER_ERROR"
        cond3 = res_token == "ERROR: CONTEXT_EXCEEDED"
        grade(10, cond1 and cond2 and cond3, "Retry 횟수 제어, 토큰 체크 또는 에러 메시지 처리가 틀립니다.")
    except Exception as e:
        grade(10, False, str(e))

    # 결과 출력
    total_score = sum(scores.values())
    print("-" * 50)
    print(f"{'문제':<10} | {'점수':<5} | {'상태'}")
    print("-" * 50)
    for i in range(1, 11):
        print(f"문제 {i:02d}    | {scores.get(i, 0):<5} | {feedback.get(i, 'N/A')}")
    print("-" * 50)
    print(f"최종 총점: {total_score} / 100")
    print("-" * 50)

# 이 코드는 단독 실행 시 학생의 정답지(solution.py)를 불러와 테스트합니다.
if __name__ == "__main__":
    try:
        import solution as student_code
        run_grader(student_code)
    except ImportError:
        print("에러: 'solution.py' 파일을 찾을 수 없습니다.")